In [9]:
import pandas as pd
import geopandas as gpd
import shapely.wkb as wkb

def read_select(path, usecols=None, filters=None, xy_cols=None, wkb_col=None, crs="EPSG:4326", encoding=None):
    """
    通用 CSV 读取函数，自动处理 X/Y 或 WKB 格式
    参数:
        path: str, 文件路径
        usecols: list, 要读取的列
        filters: dict, 筛选条件 {列名: 值}
        xy_cols: tuple, ("X","Y") 列名
        wkb_col: str, 存放WKB的列名，比如 "geom"
        fix_lonlat: bool, 是否修正经纬度错位 (适用于 renewable_project_nem)
        crs: 坐标参考系，默认 EPSG:4326
        encoding: str, 文件编码格式
    """
    # 尝试不同的编码格式
    encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']

    if encoding:
        encodings_to_try = [encoding] + encodings_to_try

    df = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(path, usecols=usecols, encoding=enc)
            print(f"成功使用编码: {enc}")
            break
        except UnicodeDecodeError:
            print(f"编码 {enc} 失败，尝试下一个...")
            continue

    if df is None:
        raise ValueError(f"无法使用任何编码读取文件: {path}")

    # 筛选
    if filters:
        for col, val in filters.items():
            df = df[df[col] == val]

    # 如果是 X,Y
    if xy_cols:
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[xy_cols[0]], df[xy_cols[1]]), crs=crs)

    # 如果是 geom(WKB)
    elif wkb_col:
        df["geometry"] = df[wkb_col].apply(lambda x: wkb.loads(bytes.fromhex(x), hex=True))
        gdf = gpd.GeoDataFrame(df.drop(columns=[wkb_col]), geometry="geometry", crs=crs)

    else:
        raise ValueError("必须指定 xy_cols 或 wkb_col")

    return gdf



def spatial_buffer_join(renewable_gdf, targets, buffer_radius=3000):
    """
    给定 renewable project 的位置，生成 buffer 并判定哪些目标数据被包含
    参数:
        renewable_gdf: renewable projects 的 GeoDataFrame
        targets: dict，key为名称，value为GeoDataFrame
        buffer_radius: 缓冲半径（米）
    返回:
        dict，key为名称，value为空间相交的结果
    """
    # 确保 CRS 为投影坐标（米单位），先转为 EPSG:3857
    renewable_proj = renewable_gdf.to_crs(epsg=3857)
    buffer = renewable_proj.buffer(buffer_radius)

    results = {}
    for name, gdf in targets.items():
        gdf_proj = gdf.to_crs(epsg=3857)
        join = gpd.sjoin(gdf_proj, gpd.GeoDataFrame(geometry=buffer), predicate="within")
        results[name] = join
    return results

def classify_fire_risk(row):
    """ 根据 allcount 和 yrsfrburn 生成火灾风险等级
    规则： - 低风险：allcount <=1 且 yrsfrburn >=30
    - 中风险：1<allcount<=3 或 10<=yrsfrburn<30
    - 高风险：allcount >3 或 yrsfrburn <10
    """
    if row["allcount"] <= 1 and row["yrsfrburn"] >= 30:
        return 0
    elif (1 < row["allcount"] <= 3) or (10 <= row["yrsfrburn"] < 30):
        return 1
    elif row["allcount"] > 3 or row["yrsfrburn"] < 10:
        return 2
    else:
        return None

In [17]:
# 1. 导入数据
metro = read_select("https://github.com/zhenweiw1//capstone/raw/refs/heads/main/Metro_Fire_Facilities.csv",
                    filters={"facility_state": "VICTORIA"},
                    xy_cols=("X","Y"))

rural = read_select("https://github.com/zhenweiw1//capstone/raw/refs/heads/main/Rural_Country_Fire_Service_Facilities.csv",
                    filters={"facility_state": "VICTORIA"},
                    xy_cols=("X","Y"))

bushfire_prone = read_select("https://github.com/zhenweiw1//capstone/raw/refs/heads/main/vic_fire_bushfire_prone_area.csv",
                              wkb_col="geom")

fire_history = read_select("https://github.com/zhenweiw1//capstone/raw/refs/heads/main/vic_fire_history_latest.csv",
                            usecols=["geom","allcount","yrsfrburn"],
                            wkb_col="geom")
fire_history["risk_class"] = fire_history.apply(classify_fire_risk,axis=1)

fire_manage_zone = read_select("https://github.com/zhenweiw1//capstone/raw/refs/heads/main/vic_fire_manage_zone.csv",
                                usecols=["geom","zonetype"],
                                wkb_col="geom")

renewable_project = read_select("https://github.com/zhenweiw1//capstone/raw/refs/heads/main/renewable_project_nem.csv",
                                filters={"Region":"VIC1", "Status Bucket Summary": "Existing less Announced Withdrawal"},
                                usecols=["Region","Status Bucket Summary","SurveyId","X","Y"],
                                xy_cols=("X","Y"),
                                encoding='latin-1')  # 尝试 latin-1 编码

# 2. 空间buffer分析
targets = {
    "MetroFireFacilities": metro,
    "RuralFireFacilities": rural,
    "BushfireProneArea": bushfire_prone,
    "FireHistory": fire_history,
    "FireManageZone": fire_manage_zone
}

buffer_results = spatial_buffer_join(renewable_project, targets, buffer_radius=3000)

# 3. 输出结果示例
for k, v in buffer_results.items():
    print(f"=== {k} within 3km ===")
    print(v.head())



成功使用编码: utf-8
成功使用编码: utf-8
成功使用编码: utf-8
成功使用编码: utf-8
成功使用编码: utf-8
成功使用编码: latin-1
=== MetroFireFacilities within 3km ===
            X          Y  objectid         featuretype  \
2  144.960202 -37.827018         3  EMERGENCY FACILITY   
2  144.960202 -37.827018         3  EMERGENCY FACILITY   
2  144.960202 -37.827018         3  EMERGENCY FACILITY   
2  144.960202 -37.827018         3  EMERGENCY FACILITY   
2  144.960202 -37.827018         3  EMERGENCY FACILITY   

                                          descripton                class  \
2  An emergency management facility from which a ...  METRO FIRE FACILITY   
2  An emergency management facility from which a ...  METRO FIRE FACILITY   
2  An emergency management facility from which a ...  METRO FIRE FACILITY   
2  An emergency management facility from which a ...  METRO FIRE FACILITY   
2  An emergency management facility from which a ...  METRO FIRE FACILITY   

         facility_name facility_operationalstatus facility_addr

In [18]:
metro

,X,Y,objectid,featuretype,descripton,class,facility_name,facility_operationalstatus,facility_address,abs_suburb,...,gnaf_building_name,gnaf_address_detail_pid,gnaf_formatted_address,gnaf_confidence,gnaf_postcode,gnaf_suburb,distance_to_gnaf,gnaf_lat,gnaf_long,geometry
0,144.995009,-37.744618,1,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,PRESTON MFB,OPERATIONAL,NaN,PRESTON,...,NaN,GAVIC420660529,471 BELL STREET,1.0,3072.0,PRESTON,0.0,-37.744618,144.995008,POINT (144.99501 -37.74462)
1,145.239048,-37.810372,2,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,RINGWOOD MFB,OPERATIONAL,NaN,RINGWOOD EAST,...,NaN,GAVIC420535858,270-274 MAROONDAH HIGHWAY,2.0,3134.0,RINGWOOD,0.0,-37.810371,145.239048,POINT (145.23905 -37.81037)
2,144.960202,-37.827018,3,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,SOUTH MELBOURNE MFB,OPERATIONAL,NaN,SOUTHBANK,...,MFB STATION NO.38,GAVIC419831406,26-40 MORAY STREET,1.0,3006.0,SOUTHBANK,0.0,-37.827018,144.960202,POINT (144.9602 -37.82702)
3,144.814066,-37.729745,4,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,ST. ALBANS MFB,OPERATIONAL,NaN,ST ALBANS,...,NaN,GAVIC421710136,5-9 TAYLORS ROAD,1.0,3021.0,ST ALBANS,0.0,-37.729745,144.814066,POINT (144.81407 -37.72974)
4,144.829388,-37.774654,5,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,SUNSHINE MFB,OPERATIONAL,NaN,SUNSHINE NORTH,...,NaN,GAVIC412791314,30-32 MCINTYRE ROAD,1.0,3020.0,SUNSHINE NORTH,0.0,-37.774654,144.829388,POINT (144.82939 -37.77465)
5,144.817615,-37.815981,6,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,NORTH LAVERTON MFB,OPERATIONAL,NaN,SUNSHINE WEST,...,NaN,GAVIC424060097,88 BOUNDARY ROAD,1.0,3020.0,SUNSHINE WEST,0.0,-37.815981,144.817615,POINT (144.81761 -37.81598)
6,144.775141,-37.695719,7,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,TAYLORS LAKES MFB,OPERATIONAL,NaN,TAYLORS LAKES,...,NaN,GAVIC421136372,470 MELTON HIGHWAY,1.0,3038.0,TAYLORS LAKES,0.0,-37.695719,144.775140,POINT (144.77514 -37.69572)
7,145.133844,-37.762398,8,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,TEMPLESTOWE MFB,OPERATIONAL,NaN,TEMPLESTOWE,...,NaN,GAVIC420007791,40-42 SERPELLS ROAD,2.0,3106.0,TEMPLESTOWE,0.0,-37.762397,145.133844,POINT (145.13384 -37.7624)
8,145.164476,-37.903581,9,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,GLEN WAVERLEY MFB,OPERATIONAL,NaN,GLEN WAVERLEY,...,NaN,GAVIC413595802,645 FERNTREE GULLY ROAD,0.0,3150.0,GLEN WAVERLEY,0.0,-37.903581,145.164476,POINT (145.16448 -37.90358)
9,145.098686,-37.704320,10,EMERGENCY FACILITY,An emergency management facility from which a ...,METRO FIRE FACILITY,GREENSBOROUGH MFB,OPERATIONAL,NaN,GREENSBOROUGH,...,NaN,GAVIC423383247,141 GRIMSHAW STREET,0.0,3088.0,GREENSBOROUGH,0.0,-37.704320,145.098686,POINT (145.09869 -37.70432)


In [13]:
# 核心表
Projects (
    project_id INTEGER PRIMARY KEY,
    name TEXT,
    x REAL,
    y REAL,
    buffer_radius REAL DEFAULT 3000
)

SpatialFeatures (
    feature_id INTEGER PRIMARY KEY,
    project_id INTEGER,
    feature_type TEXT,   -- e.g., "Metro_Fire_Facility", "Bushfire_Prone_Area", "Fire_History", "Manage_Zone"
    feature_geom BLOB,   -- 存储 WKB
    attributes JSON,     -- 存储原始属性（如 zonetype, allcount, yrsfrburn）
    FOREIGN KEY(project_id) REFERENCES Projects(project_id)
)

SpatialFeatures (
    feature_id INTEGER PRIMARY KEY,
    project_id INTEGER,
    feature_type TEXT,   -- e.g., "Metro_Fire_Facility", "Bushfire_Prone_Area", "Fire_History", "Manage_Zone"
    feature_geom BLOB,   -- 存储 WKB
    attributes JSON,     -- 存储原始属性（如 zonetype, allcount, yrsfrburn）
    FOREIGN KEY(project_id) REFERENCES Projects(project_id)
)

RiskLabels (
    label_id INTEGER PRIMARY KEY,
    project_id INTEGER,
    allcount INTEGER,
    yrsfrburn INTEGER,
    risk_level INTEGER,   -- 0=低, 1=中, 2=高
    FOREIGN KEY(project_id) REFERENCES Projects(project_id)
)

SyntaxError: invalid character '（' (U+FF08) (ipython-input-2648932107.py, line 15)